# Kapitel 7: Bygning af chatapplikationer
## OpenAI API Kom godt i gang

Denne notesbog er tilpasset fra [Azure OpenAI Samples Repository](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst), som inkluderer notesbøger, der får adgang til [Azure OpenAI](notebook-azure-openai.ipynb) tjenester.

Python OpenAI API fungerer også med Azure OpenAI-modeller med et par ændringer. Lær mere om forskellene her: [Hvordan man skifter mellem OpenAI og Azure OpenAI endepunkter med Python](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/switching-endpoints?WT.mc_id=academic-109527-jasmineg)


# Oversigt  
"Store sprogmodeller er funktioner, der kortlægger tekst til tekst. Givet en indgangsstreng af tekst, prøver en stor sprogmodel at forudsige den tekst, der kommer næste gang"(1). Denne "quickstart"-notebook vil introducere brugere til højniveau LLM-koncepter, kernepakke-krav for at komme i gang med AML, en blid introduktion til promptdesign og flere korte eksempler på forskellige anvendelsestilfælde. 


## Indholdsfortegnelse  

[Oversigt](#overview)  
[Sådan bruger du OpenAI Service](#how-to-use-openai-service)  
[1. Oprettelse af din OpenAI Service](#1.-creating-your-openai-service)  
[2. Installation](#2.-installation)    
[3. Legitimation](#3.-credentials)  

[Brugstilfælde](#use-cases)    
[1. Opsummér Tekst](#1.-summarize-text)  
[2. Klassificér Tekst](#2.-classify-text)  
[3. Generér Nye Produktnavne](#3.-generate-new-product-names)  
[4. Finjuster en Klassificerer](#4.fine-tune-a-classifier)  

[Referencer](#references)


### Byg din første prompt  
Denne korte øvelse vil give en grundlæggende introduktion til at indsende prompts til en OpenAI-model for en enkel opgave "sammenfatning".


**Trin**:  
1. Installer OpenAI-biblioteket i dit Python-miljø  
2. Indlæs standard hjælpebiblioteker og sæt dine typiske OpenAI sikkerhedsoplysninger til den OpenAI-tjeneste, du har oprettet  
3. Vælg en model til din opgave  
4. Opret en simpel prompt til modellen  
5. Send din forespørgsel til model-API'et!


### 1. Installer OpenAI


In [ ]:
%pip install openai python-dotenv

### 2. Importér hjælpebiblioteker og opret legitimationsoplysninger


In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("OPENAI_API_KEY","")
assert API_KEY, "ERROR: OpenAI Key is missing"

client = OpenAI(
    api_key=API_KEY
    )


### 3. At finde den rigtige model  
Modeller som GPT-4o og GPT-4o mini kan forstå og generere naturligt sprog, og er tilgængelige på OpenAI-platformen med forskellige niveauer af kraft og hastighed, der passer til forskellige opgaver.


In [ ]:
# Select a general purpose chat model
model = "gpt-5-mini"


## 4. Prompt Design  

"Magien ved store sprogmodeller er, at ved at blive trænet til at minimere denne forudsigelsesfejl over enorme mængder tekst, ender modellerne med at lære begreber, der er nyttige til disse forudsigelser. For eksempel lærer de begreber som"(1):

* hvordan man staver
* hvordan grammatik fungerer
* hvordan man omformulerer
* hvordan man besvarer spørgsmål
* hvordan man fører en samtale
* hvordan man skriver på mange sprog
* hvordan man koder
* osv.

#### Hvordan man styrer en stor sprogmodel  
"Af alle input til en stor sprogmodel er tekstprompten langt den mest indflydelsesrige(1).

Store sprogmodeller kan promptes til at producere output på nogle få måder:

Instruktion: Fortæl modellen, hvad du vil have
Fuldførelse: Fremkald modellen til at fuldføre begyndelsen af det, du ønsker
Demonstration: Vis modellen, hvad du vil have, enten med:
Et par eksempler i prompten
Mange hundrede eller tusinder af eksempler i et fintunings træningsdatasæt"



#### Der er tre grundlæggende retningslinjer for at skabe prompts:

**Vis og fortæl**. Gør det klart, hvad du ønsker, enten gennem instruktioner, eksempler eller en kombination af begge. Hvis du vil have modellen til at rangordne en liste af ting i alfabetisk rækkefølge eller til at klassificere et afsnit efter sentiment, så vis den, at det er det, du ønsker.

**Giv kvalitetsdata**. Hvis du prøver at bygge en klassifikator eller få modellen til at følge et mønster, så sørg for, at der er nok eksempler. Sørg for at korrekturlæse dine eksempler — modellen er som regel smart nok til at gennemskue basale stavefejl og give dig et svar, men den kan også antage, at det er tilsigtet, og det kan påvirke svaret.

**Tjek dine indstillinger.** Temperature- og top_p-indstillingerne styrer, hvor deterministisk modellen er i at generere et svar. Hvis du beder om et svar, hvor der kun er ét rigtigt svar, vil du gerne sætte disse lavere. Hvis du leder efter mere forskelligartede svar, kan du sætte dem højere. Den mest almindelige fejl folk laver med disse indstillinger er at antage, at det er "snilde" eller "kreativitet" kontroller.


Kilde: https://learn.microsoft.com/azure/ai-foundry/openai/overview


### 5. Indsend!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


### Gentag det samme kald, hvordan sammenlignes resultaterne?


In [ ]:

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},],
  store=False,)

response.output_text


## Opsummer Tekst  
#### Udfordring  
Opsummer tekst ved at tilføje 'tl;dr:' til slutningen af et tekstafsnit. Bemærk, hvordan modellen forstår at udføre en række opgaver uden yderligere instruktioner. Du kan eksperimentere med mere beskrivende prompts end tl;dr for at ændre modellens adfærd og tilpasse den opsummering, du modtager(3).  

Nyere arbejde har vist betydelige fremskridt på mange NLP-opgaver og benchmarks ved forudtræning på en stor tekstkorpus efterfulgt af finjustering på en specifik opgave. Selvom arkitekturen typisk er opgave-agnostisk, kræver denne metode stadig opgavespecifikke finjusteringsdatasæt med tusinder eller titusinder af eksempler. Til sammenligning kan mennesker normalt udføre en ny sprogopgave ud fra kun få eksempler eller simple instruktioner – noget som nuværende NLP-systemer stadig i høj grad har svært ved. Her viser vi, at skalering af sprogmodeller i høj grad forbedrer opgave-agnostisk, few-shot ydeevne, nogle gange endda opnår konkurrenceevne med tidligere state-of-the-art finjusteringsmetoder.  



Tl;dr


# Øvelser for flere brugssituationer  
1. Opsummér tekst  
2. Klassificér tekst  
3. Generér nye produktnavne


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Klassificer tekst  
#### Udfordring  
Klassificer elementer i kategorier, der gives ved inferens. I det følgende eksempel giver vi både kategorierne og teksten, der skal klassificeres, i prompten(*playground_reference). 

Kundeservicehenvendelse: Hej, en af tasterne på mit bærbare tastatur gik i stykker for nylig, og jeg får brug for en erstatning:

Klassificeret kategori:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},],
  store=False,)

response.output_text


## Generer Nye Produktnavne
#### Udfordring
Skab produktnavne ud fra eksempler på ord. Her inkluderer vi i prompten information om produktet, vi skal generere navne til. Vi giver også et lignende eksempel for at vise det mønster, vi ønsker at modtage. Vi har også sat temperaturværdien højt for at øge tilfældigheden og få mere innovative svar.

Produktbeskrivelse: En hjemmemilkshake-maskine
Frøord: hurtig, sund, kompakt.
Produktnavne: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Produktbeskrivelse: Et par sko, der kan passe til enhver fodstørrelse.
Frøord: tilpasningsdygtig, pasform, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.responses.create(
  model=model,
  input = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}],
  store=False,)

response.output_text


# Referencer  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Bedste praksis for finjustering af GPT-modeller til at klassificere tekst](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# For Mere Hjælp  
[OpenAI Commercialization Team](AzureOpenAITeam@microsoft.com) 


# Bidragydere
* Louis Li  


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er blevet oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiserede oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritisk information anbefales professionel menneskelig oversættelse. Vi påtager os intet ansvar for misforståelser eller fejltolkninger, der opstår som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
